In [1]:
import os, shutil, zipfile, yaml
from pathlib import Path

DETECTION_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        DETECTION_ROOT = root
        break
print("Detected shared dataset root:", DETECTION_ROOT)

WORK_DIR = Path('/kaggle/working/dental_final')

for s in ['train', 'val', 'test']:
    img_out = WORK_DIR / s / 'images'
    lbl_out = WORK_DIR / s / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)

    zip_path = Path(DETECTION_ROOT) / f'{s}_images.zip'
    extracted_dir = Path(DETECTION_ROOT) / f'{s}_images'

    if zip_path.exists():
        print(f"{s}: found zip, extracting...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(img_out)
    elif extracted_dir.exists():
        print(f"{s}: zip already auto-extracted by Kaggle, copying folder...")
        for f in extracted_dir.glob('*.jpg'):
            shutil.copy2(f, img_out / f.name)
    else:
        print(f"{s}: NEITHER zip nor extracted folder found — check the listing above for the real name/location")
        continue

    shutil.copytree(Path(DETECTION_ROOT) / s / 'labels', lbl_out, dirs_exist_ok=True)

    n_img = len(list(img_out.glob('*.jpg')))
    n_lbl = len(list(lbl_out.glob('*.txt')))
    print(f"{s}: {n_img} images, {n_lbl} labels\n")

with open(Path(DETECTION_ROOT) / 'data.yaml') as f:
    src_yaml = yaml.safe_load(f)

final_yaml = {
    'train': str(WORK_DIR / 'train' / 'images'),
    'val': str(WORK_DIR / 'val' / 'images'),
    'test': str(WORK_DIR / 'test' / 'images'),
    'nc': src_yaml['nc'],
    'names': src_yaml['names'],
}
with open(WORK_DIR / 'data.yaml', 'w') as f:
    yaml.dump(final_yaml, f)

print("data.yaml ready at:", WORK_DIR / 'data.yaml')

Detected shared dataset root: /kaggle/input/datasets/redwanahmed2025/dental-detection-final-v3/dental_final
train: zip already auto-extracted by Kaggle, copying folder...
train: 7998 images, 7998 labels

val: zip already auto-extracted by Kaggle, copying folder...
val: 993 images, 993 labels

test: zip already auto-extracted by Kaggle, copying folder...
test: 1009 images, 1009 labels

data.yaml ready at: /kaggle/working/dental_final/data.yaml


# CSE 445 — Assignment 1 — Notebook 2: YOLOv10 Training, Evaluation & Error Analysis
Group: [your group number]
Members: [names + IDs]
Dataset: Dental OPG Object Detection Dataset (same corrected/split version as Notebook 1)
Model: YOLOv10-m

In [2]:
CONFIG = {
    'model_variant': 'yolov10m.pt',
    'model_size_justification': (
        "Dataset has 10,000 images (8k-20k bucket -> 'm or l' per guidance table). "
        "'m' chosen over 'l' for faster, timeout-safe training given the 2-day deadline, "
        "and to keep model capacity identical across v10/v12/v26 for a fair comparison."
    ),
    'epochs': 50,                  # mandated minimum for YOLO
    'imgsz': 640,                  # dataset's confirmed native resolution
    'batch': 16,                   # assignment's own stated safe value for T4 @ 640
    'optimizer': 'SGD',
    'lr0': 0.01,                   # matches what 'auto' would select at ~25,000 iterations
    'cos_lr': True,
    'warmup_epochs': 3,
    'patience': 50,                # == epochs, deliberately avoids conflicting with the
                                    # mandated minimum-50-epoch requirement (see reasoning above)

    # Augmentation - carried over from Notebook 1's measured/verified config
    'hsv_h': 0.0,
    'hsv_s': 0.0,
    'hsv_v': 0.4,
    'fliplr': 0.5,
    'flipud': 0.0,
    'degrees': 10,
    'translate': 0.1,
    'scale': 0.3,
    'mosaic': 1.0,
    'close_mosaic': 10,
    'overlap_mask': True,
    'copy_paste': 0.0,

    # Inference / evaluation
    'iou': 0.6,   # lowered from default 0.7 per dense-scene guidance; moderate choice given
                  # our EDA also found legitimate close-together (non-duplicate) findings -
                  # to be checked specifically in error analysis
    'conf': 0.25,
}

print("YOLOv10 CONFIG set:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

YOLOv10 CONFIG set:
  model_variant: yolov10m.pt
  model_size_justification: Dataset has 10,000 images (8k-20k bucket -> 'm or l' per guidance table). 'm' chosen over 'l' for faster, timeout-safe training given the 2-day deadline, and to keep model capacity identical across v10/v12/v26 for a fair comparison.
  epochs: 50
  imgsz: 640
  batch: 16
  optimizer: SGD
  lr0: 0.01
  cos_lr: True
  warmup_epochs: 3
  patience: 50
  hsv_h: 0.0
  hsv_s: 0.0
  hsv_v: 0.4
  fliplr: 0.5
  flipud: 0.0
  degrees: 10
  translate: 0.1
  scale: 0.3
  mosaic: 1.0
  close_mosaic: 10
  overlap_mask: True
  copy_paste: 0.0
  iou: 0.6
  conf: 0.25


In [3]:
!pip install ultralytics -q

import torch, os
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

DETECTION_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        DETECTION_ROOT = root
        break
print("Detected shared dataset root:", DETECTION_ROOT)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.7 MB/s eta 0:00:00
GPU available: True
GPU name: Tesla T4
Detected shared dataset root: /kaggle/input/datasets/redwanahmed2025/dental-detection-final-v3/dental_final


In [4]:
import os, shutil, zipfile, yaml
from pathlib import Path

DETECTION_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        DETECTION_ROOT = root
        break
print("Detected shared dataset root:", DETECTION_ROOT)

WORK_DIR = Path('/kaggle/working/dental_final')

for s in ['train', 'val', 'test']:
    img_out = WORK_DIR / s / 'images'
    lbl_out = WORK_DIR / s / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)

    zip_path = Path(DETECTION_ROOT) / f'{s}_images.zip'
    extracted_dir = Path(DETECTION_ROOT) / f'{s}_images'

    if zip_path.exists():
        print(f"{s}: found zip, extracting...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(img_out)
    elif extracted_dir.exists():
        print(f"{s}: zip already auto-extracted by Kaggle, copying folder...")
        for f in extracted_dir.glob('*.jpg'):
            shutil.copy2(f, img_out / f.name)
    else:
        print(f"{s}: NEITHER zip nor extracted folder found — check the listing above for the real name/location")
        continue

    shutil.copytree(Path(DETECTION_ROOT) / s / 'labels', lbl_out, dirs_exist_ok=True)

    n_img = len(list(img_out.glob('*.jpg')))
    n_lbl = len(list(lbl_out.glob('*.txt')))
    print(f"{s}: {n_img} images, {n_lbl} labels\n")

with open(Path(DETECTION_ROOT) / 'data.yaml') as f:
    src_yaml = yaml.safe_load(f)

final_yaml = {
    'train': str(WORK_DIR / 'train' / 'images'),
    'val': str(WORK_DIR / 'val' / 'images'),
    'test': str(WORK_DIR / 'test' / 'images'),
    'nc': src_yaml['nc'],
    'names': src_yaml['names'],
}
with open(WORK_DIR / 'data.yaml', 'w') as f:
    yaml.dump(final_yaml, f)

print("data.yaml ready at:", WORK_DIR / 'data.yaml')

Detected shared dataset root: /kaggle/input/datasets/redwanahmed2025/dental-detection-final-v3/dental_final
train: zip already auto-extracted by Kaggle, copying folder...
train: 7998 images, 7998 labels

val: zip already auto-extracted by Kaggle, copying folder...
val: 993 images, 993 labels

test: zip already auto-extracted by Kaggle, copying folder...
test: 1009 images, 1009 labels

data.yaml ready at: /kaggle/working/dental_final/data.yaml


1-epoch dry run

In [5]:
from ultralytics import YOLO
import time

model = YOLO(CONFIG['model_variant'])

start = time.time()
dry_run_results = model.train(
    data=str(WORK_DIR / 'data.yaml'),
    epochs=1,
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    optimizer=CONFIG['optimizer'],
    lr0=CONFIG['lr0'],
    cos_lr=CONFIG['cos_lr'],
    warmup_epochs=CONFIG['warmup_epochs'],
    hsv_h=CONFIG['hsv_h'], hsv_s=CONFIG['hsv_s'], hsv_v=CONFIG['hsv_v'],
    fliplr=CONFIG['fliplr'], flipud=CONFIG['flipud'], degrees=CONFIG['degrees'],
    translate=CONFIG['translate'], scale=CONFIG['scale'],
    mosaic=CONFIG['mosaic'], close_mosaic=CONFIG['close_mosaic'],
    overlap_mask=CONFIG['overlap_mask'], copy_paste=CONFIG['copy_paste'],
    project='/kaggle/working/yolov10m_dental',
    name='dry_run',
    exist_ok=True,
)
elapsed_min = (time.time() - start) / 60

print(f"\n1-epoch time: {elapsed_min:.1f} minutes")
print(f"Estimated total for 50 epochs: {elapsed_min * 50 / 60:.1f} hours")
print(f"Kaggle session budget: ~8 hours (leaving 1 hour buffer)")
print(f"Fits within budget: {elapsed_min * 50 / 60 < 8}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dental_final/data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hs

 full training run

In [6]:
model_full = YOLO(CONFIG['model_variant'])  # fresh pretrained weights, not the dry-run's partially-trained ones

full_results = model_full.train(
    data=str(WORK_DIR / 'data.yaml'),
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    optimizer=CONFIG['optimizer'],
    lr0=CONFIG['lr0'],
    cos_lr=CONFIG['cos_lr'],
    warmup_epochs=CONFIG['warmup_epochs'],
    patience=CONFIG['patience'],
    hsv_h=CONFIG['hsv_h'], hsv_s=CONFIG['hsv_s'], hsv_v=CONFIG['hsv_v'],
    fliplr=CONFIG['fliplr'], flipud=CONFIG['flipud'], degrees=CONFIG['degrees'],
    translate=CONFIG['translate'], scale=CONFIG['scale'],
    mosaic=CONFIG['mosaic'], close_mosaic=CONFIG['close_mosaic'],
    overlap_mask=CONFIG['overlap_mask'], copy_paste=CONFIG['copy_paste'],
    project='/kaggle/working/yolov10m_dental',
    name='full_run',
    exist_ok=True,
)

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dental_final/data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=full_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_